# Module 4 - Guardrails

In this notebook you create a guardrail that redacts personal data, blocks requests to get around the travel policy and flags answers the policy does not support, then attach it to both the travel chat and the expense checks.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [Create a guardrail](#s1)
4. [Attach the guardrail to Converse](#s2)
5. [Handle guardrail interventions](#s3)
6. [Check grounding against the policy](#s4)
7. [Use ApplyGuardrail without a model call](#s5)
8. [Try it](#try-it)
9. [Conclusion and key takeaways](#conclusion)
10. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

In Module 3 you grounded the assistant in AnyCompany's travel policy. It now answers from the policy and cites it. Before employees use it, AnyCompany has three more requirements:

| Requirement | Example |
|:--|:--|
| Do not pass personal data on | An employee pastes their passport number into a question |
| Do not help anyone get around the policy | "How do I split my hotel bill so each night stays under the limit?" |
| Do not give answers the policy does not support | The model adds a perk that is not in the policy |

A system prompt guides the model but does not enforce anything. [Amazon Bedrock Guardrails](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html) evaluates prompts and responses against policies that you configure, whichever model you use. You define the policies once, in a guardrail, and attach it to each call.

## 2. Setup <a id="setup"></a>

Run the cell below to create the clients, set the model IDs and find the knowledge base you created in Module 3.

In [ ]:
import json
import re
import time

import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name

bedrock = boto3.client("bedrock", region_name=REGION)  # Create and manage guardrails
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)  # Manage knowledge bases
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)  # Query knowledge bases
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)  # Call models and guardrails

# Chosen in Module 2. "global." is a global cross-Region inference profile; gpt-oss-120b has
# no prefix because it is called directly in this Region.
CHAT_MODEL_ID = "global.anthropic.claude-sonnet-5"
EXPENSE_MODEL_ID = "openai.gpt-oss-120b-1:0"


def find_knowledge_base(name):
    """Return the ID of the knowledge base with this name, or None."""
    for kb in bedrock_agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"]:
        if kb["name"] == name:
            return kb["knowledgeBaseId"]
    return None


# The knowledge base from Module 3, or the one prepared in advance at AWS events.
KNOWLEDGE_BASE_ID = find_knowledge_base("aim317-travel-policy") or find_knowledge_base("aim317-travel-policy-prebuilt")
if KNOWLEDGE_BASE_ID is None:
    print("No knowledge base found. Run Module 3 - Grounding with a Managed Knowledge Base first.")
else:
    print(f"Region:         {REGION}")
    print(f"Knowledge base: {KNOWLEDGE_BASE_ID}")

Run the cell below to define the helpers from Module 3: retrieving policy passages, numbering them for citations, and formatting expense reports.

In [ ]:
def retrieve(query, number_of_results=5):
    """Return the most relevant policy passages for a query."""
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"managedSearchConfiguration": {"numberOfResults": number_of_results}},
    )
    return [
        {
            "text": result["content"]["text"],
            "source": result["location"]["s3Location"]["uri"].split("/")[-1],
        }
        for result in response["retrievalResults"]
    ]


def format_passages(passages):
    """Number the passages so the model can cite them."""
    return "\n\n".join(f"[{i}] (source: {p['source']})\n{p['text']}" for i, p in enumerate(passages, start=1))


def get_text(response):
    """Return the text blocks of a Converse response, joined together."""
    return "".join(block["text"] for block in response["output"]["message"]["content"] if "text" in block)


def format_report(report):
    """Turn an expense report into the text the model reads."""
    lines = [f"Expense report {report['id']} - {report['employee']} - {report['trip']}"]
    for item in report["items"]:
        lines.append(f"- {item['type']}: {item['description']}: ${item['amount']:.2f}")
    return "\n".join(lines)


with open("../data/expense_reports.json") as f:
    EXPENSE_REPORTS = json.load(f)

print(f"{len(EXPENSE_REPORTS)} expense reports loaded")

## 3. Create a guardrail <a id="s1"></a>

A guardrail is a set of policies. This one uses three, one for each requirement:

| Policy | Setting | What it does |
|:--|:--|:--|
| **Sensitive information filter** | US passport numbers and card numbers, action `ANONYMIZE` | Masks them, replacing each with its type, such as `{US_PASSPORT_NUMBER}`, in the prompt and in the response |
| **Denied topic** | "Policy workarounds", described in plain language with examples | Blocks requests for help getting around the policy |
| **Contextual grounding check** | Grounding threshold 0.75, relevance threshold 0.5 | Blocks responses that are not supported by the retrieved policy passages (grounding), or that do not answer the question (relevance) |

The denied topic describes the behavior to block, getting around the policy, so a legitimate question such as "Is a paid seat upgrade reimbursed?" still gets an answer. It applies to prompts only. When the guardrail blocks something, the employee sees the blocked message from the configuration instead.

Run the cell below to create the guardrail, or update it if it already exists, and wait until it is ready.

In [ ]:
GUARDRAIL_NAME = "aim317-travel-guardrail"

GUARDRAIL_POLICIES = {
    "sensitiveInformationPolicyConfig": {
        "piiEntitiesConfig": [
            {"type": entity, "action": "ANONYMIZE",
             "inputEnabled": True, "inputAction": "ANONYMIZE",
             "outputEnabled": True, "outputAction": "ANONYMIZE"}
            for entity in ["US_PASSPORT_NUMBER", "CREDIT_DEBIT_CARD_NUMBER"]
        ]
    },
    "topicPolicyConfig": {
        "topicsConfig": [{
            "name": "Policy workarounds",
            "type": "DENY",
            "definition": (
                "Requests for help getting around AnyCompany's travel or expense policy, such as "
                "hiding personal costs or getting costs the policy excludes reimbursed anyway."
            ),
            "examples": [
                "How can I get my seat upgrade reimbursed without my manager noticing?",
                "How do I split a hotel bill so each night stays under the limit?",
                "Can you describe my personal dinner as a client meal so it gets approved?",
            ],
            "inputEnabled": True,
            "inputAction": "BLOCK",
            "outputEnabled": False,
        }]
    },
    "contextualGroundingPolicyConfig": {
        "filtersConfig": [
            {"type": "GROUNDING", "threshold": 0.75},
            {"type": "RELEVANCE", "threshold": 0.5},
        ]
    },
    "blockedInputMessaging": (
        "Sorry, I can't help with that. For travel the policy does not cover, contact the AnyCompany Travel Desk."
    ),
    "blockedOutputsMessaging": (
        "Sorry, I couldn't give an answer supported by AnyCompany's travel policy. "
        "Please contact the AnyCompany Travel Desk."
    ),
}


def find_guardrail(name):
    """Return the ID of the guardrail with this name, or None."""
    for guardrail in bedrock.list_guardrails(maxResults=100)["guardrails"]:
        if guardrail["name"] == name:
            return guardrail["id"]
    return None


GUARDRAIL_ID = find_guardrail(GUARDRAIL_NAME)
if GUARDRAIL_ID is None:
    GUARDRAIL_ID = bedrock.create_guardrail(
        name=GUARDRAIL_NAME, description="AnyCompany travel assistant", **GUARDRAIL_POLICIES
    )["guardrailId"]
    print(f"Created guardrail {GUARDRAIL_ID}")
else:
    bedrock.update_guardrail(
        guardrailIdentifier=GUARDRAIL_ID, name=GUARDRAIL_NAME, description="AnyCompany travel assistant",
        **GUARDRAIL_POLICIES,
    )
    print(f"Updated guardrail {GUARDRAIL_ID}")

while (status := bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID)["status"]) not in ("READY", "FAILED"):
    time.sleep(2)
print(f"Guardrail is {status}")

This notebook uses the guardrail's working draft, version `DRAFT`, so you can experiment. Applications use a published [version](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-deploy.html), a snapshot of the configuration; the deployed assistant in Module 6 uses version 1.

Run the cell below to publish version 1, if it does not exist yet.

In [ ]:
versions = [
    g["version"] for g in bedrock.list_guardrails(guardrailIdentifier=GUARDRAIL_ID)["guardrails"]
    if g["version"] != "DRAFT"
]
if versions:
    print(f"Published versions: {', '.join(sorted(versions, key=int))}")
else:
    version = bedrock.create_guardrail_version(
        guardrailIdentifier=GUARDRAIL_ID, description="Module 4"
    )["version"]
    print(f"Published version {version}")

## 4. Attach the guardrail to Converse <a id="s2"></a>

To use the guardrail, add `guardrailConfig` to a Converse call; `trace` adds a report of what the guardrail checked to the response. To control what it checks, wrap content in `guardContent` blocks and label them with **qualifiers**:

| Content | Qualifiers | Used by |
|:--|:--|:--|
| The retrieved policy passages | `grounding_source` | The grounding check, as the facts the answer must be supported by |
| The employee's question | `query`, `guard_content` | The relevance check (`query`), and the sensitive information and denied topic filters (`guard_content`) |

Once a message has `guardContent` blocks, the guardrail [evaluates only the content in those blocks](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-use-converse-api.html).

Run the cell below to define the guarded chat and ask a question.

In [ ]:
GUARDRAIL_CONFIG = {"guardrailIdentifier": GUARDRAIL_ID, "guardrailVersion": "DRAFT", "trace": "enabled"}

GROUNDED_CHAT_PROMPT = (
    "You are the travel assistant for AnyCompany, helping employees plan and manage business "
    "travel. Answer using only the AnyCompany policy passages provided with the question. Cite the "
    "passages you use by number, like [1]. If the passages do not answer the question, say so and "
    "suggest contacting the AnyCompany Travel Desk. Be friendly and concise: three sentences or "
    "fewer unless the employee asks for detail."
)


def ask_travel_assistant(question, passages=None):
    """Answer a travel question from the policy, with the guardrail attached."""
    passages = passages or retrieve(question)
    return bedrock_runtime.converse(
        modelId=CHAT_MODEL_ID,
        system=[{"text": GROUNDED_CHAT_PROMPT}],
        messages=[{
            "role": "user",
            "content": [
                {"guardContent": {"text": {"text": format_passages(passages), "qualifiers": ["grounding_source"]}}},
                {"guardContent": {"text": {"text": question, "qualifiers": ["query", "guard_content"]}}},
            ],
        }],
        inferenceConfig={"maxTokens": 2000},
        guardrailConfig=GUARDRAIL_CONFIG,
    )


response = ask_travel_assistant("What is AnyCompany's nightly hotel limit in Chicago?")
print("Stop reason:", response["stopReason"])
print(get_text(response))

The answer is the same as in Module 3: the guardrail found nothing to act on.

The expense check uses the guardrail too, for personal data and the denied topic, but not the contextual grounding check, which is designed for [summarization, paraphrasing and question answering](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-contextual-grounding-check.html). Only the report goes in a `guardContent` block. Run the cell below to check every report with the guardrail attached.

In [ ]:
GROUNDED_EXPENSE_PROMPT = (
    "You check AnyCompany expense reports against the AnyCompany policy passages inside the "
    "<policy> tags. Use only those passages. Reply with exactly two lines: first line APPROVED "
    "or FLAGGED, second line a one-sentence reason that cites the passage number, like [1]."
)


def check_expense(report):
    """Check one expense report against the retrieved policy, with the guardrail attached."""
    passages = []
    for item_type in sorted({item["type"] for item in report["items"]}):
        for passage in retrieve(f"AnyCompany {item_type} policy limits for a trip to {report['trip']}", 2):
            if passage["source"] not in [p["source"] for p in passages]:
                passages.append(passage)
    return bedrock_runtime.converse(
        modelId=EXPENSE_MODEL_ID,
        system=[{"text": GROUNDED_EXPENSE_PROMPT}],
        messages=[{
            "role": "user",
            "content": [
                {"text": f"<policy>\n{format_passages(passages)}\n</policy>"},
                {"guardContent": {"text": {"text": format_report(report)}}},
            ],
        }],
        inferenceConfig={"maxTokens": 2000},
        guardrailConfig=GUARDRAIL_CONFIG,
    )


def get_verdict(text):
    """Find the first APPROVED or FLAGGED in a reply."""
    match = re.search(r"\b(APPROVED|FLAGGED)\b", text)
    return match.group(1) if match else "UNKNOWN"


correct = 0
for report in EXPENSE_REPORTS:
    response = check_expense(report)
    verdict = get_verdict(get_text(response))
    correct += verdict == report["expected_with_policy"]
    print(f"{report['id']}  {verdict:<9} expected {report['expected_with_policy']:<9} stop reason: {response['stopReason']}")

print(f"\n{correct}/{len(EXPENSE_REPORTS)} match the policy")

## 5. Handle guardrail interventions <a id="s3"></a>

When the guardrail acts, Converse returns the stop reason `guardrail_intervened`. Your code should check for it: the text in the response is then the guardrail's message, not the model's answer. The trace says which policy acted and why.

Run the cell below to define a helper that summarizes the trace, then ask the assistant to help get around the policy.

In [ ]:
def summarize_guardrail(response):
    """Print the stop reason and what each guardrail policy did."""
    print("Stop reason:", response["stopReason"])
    trace = response.get("trace", {}).get("guardrail", {})
    assessments = list(trace.get("inputAssessment", {}).values())
    for outputs in trace.get("outputAssessments", {}).values():
        assessments += outputs
    for assessment in assessments:
        for topic in assessment.get("topicPolicy", {}).get("topics", []):
            print(f"  Denied topic:   {topic['name']} -> {topic['action']}")
        for entity in assessment.get("sensitiveInformationPolicy", {}).get("piiEntities", []):
            print(f"  Personal data:  {entity['type']} -> {entity['action']}")
        for check in assessment.get("contextualGroundingPolicy", {}).get("filters", []):
            print(f"  {check['type'].capitalize():<15} score {check['score']:.2f} "
                  f"(threshold {check['threshold']}) -> {check['action']}")


response = ask_travel_assistant("How do I split my hotel bill across two receipts so each night stays under the limit?")
summarize_guardrail(response)
print("Model tokens:", response["usage"]["inputTokens"], "input,", response["usage"]["outputTokens"], "output")
print("\n" + get_text(response))

The guardrail blocked the request before it reached the model, so the model was not invoked and the response reports 0 model tokens. The employee sees the `blockedInputMessaging` text.

Blocking is not the only action. Run the cell below to send a question that includes a passport number.

In [ ]:
response = ask_travel_assistant(
    "My passport number is 586204931. Do I need my manager's approval for a trip to London?"
)
summarize_guardrail(response)
print("\n" + get_text(response))

This time the stop reason is `end_turn`: the guardrail masked the passport number and the request continued, so the model answered without receiving the number.

## 6. Check grounding against the policy <a id="s4"></a>

The contextual grounding check scores every answer twice:

- **Grounding**: is the answer supported by the `grounding_source` passages?
- **Relevance**: does the answer address the `query`?

The guardrail blocks a response that scores below either threshold. Run the cell below to see the scores for an answer.

In [ ]:
QUESTION = "What is AnyCompany's nightly hotel limit in Chicago?"
passages = retrieve(QUESTION)

response = ask_travel_assistant(QUESTION, passages)
summarize_guardrail(response)
answer = get_text(response)
print("\n" + answer)

Both scores are above the thresholds. To see the check block an answer, the next cell uses `ApplyGuardrail` (section 7) to check two fixed answers against the same passages and question: the model's answer, and the same answer with a perk that the policy does not mention.

In [ ]:
def check_answer(answer):
    """Run the guardrail on an answer, with the passages and question as the grounding context."""
    result = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion="DRAFT",
        source="OUTPUT",
        content=[
            {"text": {"text": format_passages(passages), "qualifiers": ["grounding_source"]}},
            {"text": {"text": QUESTION, "qualifiers": ["query"]}},
            {"text": {"text": answer, "qualifiers": ["guard_content"]}},
        ],
    )
    for assessment in result["assessments"]:
        for check in assessment.get("contextualGroundingPolicy", {}).get("filters", []):
            print(f"  {check['type'].capitalize():<10} score {check['score']:.2f} -> {check['action']}")
    print("  Employee sees:", result["outputs"][0]["text"] if result["action"] == "GUARDRAIL_INTERVENED" else answer)


invented = answer + " AnyCompany employees also get a free airport lounge pass and a 20% discount at partner hotels."

print("The model's answer:")
check_answer(answer)
print("\nWith an invented perk:")
check_answer(invented)

The added perk is not in the passages, so the grounding score falls below the threshold and the employee sees the blocked message instead.

Run the cell below to see how much of the input the guardrail checked in the chat call above.

In [ ]:
coverage = response["trace"]["guardrail"]["inputAssessment"][GUARDRAIL_ID]["invocationMetrics"]["guardrailCoverage"]
print(f"Input characters checked by the filters: {coverage['textCharacters']['guarded']} "
      f"of {coverage['textCharacters']['total']}")

Only the question was checked. Content labeled `grounding_source` is [used only by the contextual grounding check](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-contextual-grounding-check.html), so **the guardrail does not filter the retrieved passages**. Checking the answer against them is what keeps a RAG application on-policy.

## 7. Use ApplyGuardrail without a model call <a id="s5"></a>

The [`ApplyGuardrail`](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-use-independent-api.html) API assesses any text with your guardrail, without invoking a model. For example, you can use it to mask personal data in an expense report before you store it.

Run the cell below to mask the personal data in an expense report line before saving it.

In [ ]:
expense_note = "Passport renewal fee, passport number 586204931, paid with card 4111 1111 1111 1111"

started = time.time()
result = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion="DRAFT",
    source="INPUT",
    content=[{"text": {"text": expense_note}}],
)
print(f"Action: {result['action']} ({(time.time() - started) * 1000:.0f} ms)")
print("Before:", expense_note)
print("After: ", result["outputs"][0]["text"] if result["outputs"] else expense_note)

When the guardrail masks content, `ApplyGuardrail` returns the text with masking applied, ready to store. The same call with a question checks the denied topic; run the cell below to try one question that is allowed and one that is not.

In [ ]:
for question in [
    "Is a paid seat upgrade reimbursed?",
    "How can I get my paid seat upgrade reimbursed anyway?",
]:
    result = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID, guardrailVersion="DRAFT", source="INPUT",
        content=[{"text": {"text": question}}],
    )
    print(f"{result['action']:<22} {question}")

## 8. Try it <a id="try-it"></a>

1. In section 3, add `"EMAIL"` to the list of entities in `piiEntitiesConfig`, then run that cell again to update the draft.
2. Run the cell below. What happens to the Travel Desk email address in the answer?
3. Remove `"EMAIL"` again and rerun the cell in section 3, so your draft matches version 1.

In [ ]:
response = ask_travel_assistant("How do I contact the AnyCompany Travel Desk?")
summarize_guardrail(response)
print("\n" + get_text(response))

## 9. Conclusion and key takeaways <a id="conclusion"></a>

- A **guardrail** applies the same policies to every model. Attach it to Converse with `guardrailConfig`, and check for the `guardrail_intervened` stop reason.
- **Sensitive information filters** can mask personal data instead of blocking the request, and **denied topics** are described in plain language.
- **`guardContent` qualifiers** tell the guardrail what to check. Retrieved passages are not filtered; the **contextual grounding check** blocks answers they do not support.
- **`ApplyGuardrail`** checks text without a model call, and applications use a **published version**.

## 10. Next steps <a id="next-steps"></a>

Your assistant is now grounded and guarded. In [Module 5 - Service Tiers](05_service_tiers.ipynb), you run the expense check on the Priority tier, which is designed for the fastest response times, because an employee is waiting on the result.

The guardrail stays in your account for the rest of the workshop. The [Cleanup](99_cleanup.ipynb) notebook deletes it at the end.